# MultimodalVA — Multimodal Fusion

Combine the **narrative** and the **symptom indicators** in one model. Same
`mmva.run()` call, different `task`:

- **Feature fusion** (`task="feature_fusion"`) — AutoGluon AutoMM learns jointly
  over text + tabular features.
- **Data fusion** (`task="data_fusion"`) — tabular rows are rendered to text and
  concatenated with the narrative, then a long-context transformer is fine-tuned.
- **Decision fusion** — train independent base models and combine their
  predictions: `task="voting"` (average probabilities) or `task="stacking"`
  (k-fold out-of-fold predictions → meta-learner).

> Feature fusion needs the extra: `pip install "multimodalva[feature_fusion]"`.

In [ ]:
import multimodalva as mmva
import pandas as pd

df = mmva.data("va_sample", n_per_class=40)
LABEL, TEXT = "cause_of_death", "narrative"
feature_cols = [c for c in df.columns if c.startswith("i") and c[1:4].isdigit()]
print(df.shape, "|", len(feature_cols), "features")

## 1. Feature fusion (AutoMM)

Joint text + tabular. `init_kwargs` sets constructor options like the fusion strategy and quality preset; `time_limit` caps training.

In [ ]:
# Requires: pip install "multimodalva[feature_fusion]"
ff_result = mmva.run(
    task="feature_fusion",
    data=df,
    label=LABEL,
    text_col=TEXT,
    features=feature_cols,
    model="bioclinicalbert",
    output_dir="runs/feature_fusion",
    init_kwargs={"fusion_strategy": "attention", "preset": "medium_quality"},
    time_limit=600,                # seconds; raise for better results
)
ff_result["predictions"].top1.head()

## 2. Data fusion (tabular → text, one long-context model)

No base-model lists needed — just pick a long-context model. BigBird runs natively on Apple Silicon; use Longformer on CUDA/HPC.

In [ ]:
data_fusion_result = mmva.run(
    task="data_fusion",
    data=df,
    label=LABEL,
    text_col=TEXT,
    features=feature_cols,
    model="clinicalbigbird",
    output_dir="runs/data_fusion",
    optimize=False,
    hyperparams={"epochs": 3, "batch_size": 4},
)
data_fusion_result["predictions"].top1.head()

## 3. Decision fusion — soft voting

Train independent base models and **average their probabilities**. List the base models via `text_models` / `tabular_models` (each a small spec dict).

In [ ]:
voting_result = mmva.run(
    task="voting",
    data=df,
    label=LABEL,
    text_col=TEXT,
    features=feature_cols,
    output_dir="runs/voting",
    text_models=[{"model_name": "bluebert", "max_length": 512}],
    tabular_models=[{"model_name": "lightgbm"}, {"model_name": "random_forest"}],
)
voting_result["predictions"].top1.head()

## 4. Decision fusion — stacking (super learner)

k-fold out-of-fold predictions from each base model become features for a **meta-learner**. `init_kwargs` sets the number of folds and the meta-learner.

In [ ]:
stacking_result = mmva.run(
    task="stacking",
    data=df,
    label=LABEL,
    text_col=TEXT,
    features=feature_cols,
    output_dir="runs/stacking",
    text_models=[{"model_name": "bluebert", "max_length": 512}],
    tabular_models=[{"model_name": "lightgbm"}],
    init_kwargs={"n_folds": 5,
                 "meta_learners": [{"model_name": "logistic_regression"}]},
)
stacking_result["predictions"].top1.head()

## 5. Compare all approaches

In [ ]:
from multimodalva.utils.metrics import score_predictions

runs = {
    "feature_fusion": ff_result,
    "data_fusion":    data_fusion_result,
    "voting":         voting_result,
    "stacking":       stacking_result,
}
rows = []
for name, res in runs.items():
    t = res["predictions"].top1
    rows.append({
        "model": name,
        "accuracy": score_predictions(t, "accuracy"),
        "f1_macro": score_predictions(t, "f1_macro"),
        "csmf": score_predictions(t, "csmf_accuracy"),
    })
pd.DataFrame(rows).sort_values("f1_macro", ascending=False)

Data-fusion models can be published to the Hub exactly like unimodal text models — pass `push_to_hub=True, hub_repo_id=...` to `run()` (see the getting-started notebook, section 8).